In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import spacy
from spacy import displacy

In [ ]:
def load_conll(file_path):
    sentences = []
    sentence = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip() == "":
                if sentence:
                    sentences.append(sentence)
                    sentence = []
            elif not line.startswith("-DOCSTART-"):
                parts = line.strip().split()
                if len(parts) == 4:
                    word, pos, chunk, ner = parts
                    sentence.append((word, ner))
                    
    return sentences


train_path = "/kaggle/input/conll003-englishversion/train.txt"

train_data = load_conll(train_path)

print("Number of training sentences:", len(train_data))
print("First sentence sample:", train_data[0][:10])

In [ ]:
valid_path = "/kaggle/input/conll003-englishversion/valid.txt"
test_path  = "/kaggle/input/conll003-englishversion/test.txt"

valid_data = load_conll(valid_path)
test_data  = load_conll(test_path)

print("Validation sentences:", len(valid_data))
print("Test sentences:", len(test_data))

In [ ]:
def sentence_to_text(sentence):
    return " ".join([word for word, tag in sentence])

train_texts = [sentence_to_text(s) for s in train_data]
valid_texts = [sentence_to_text(s) for s in valid_data]
test_texts  = [sentence_to_text(s) for s in test_data]

print("Sample sentence:")
print(train_texts[0])

In [ ]:
nlp_sm = spacy.load("en_core_web_sm")

print("spaCy model loaded successfully ✅")

In [ ]:
doc = nlp_sm(train_texts[0])

print("Sentence:")
print(train_texts[0])
print("\nDetected Entities:")

for ent in doc.ents:
    print(ent.text, "→", ent.label_)

In [ ]:
def extract_true_entities(sentence):
    entities = []
    entity = []
    label = None
    
    for word, tag in sentence:
        if tag.startswith("B-"):
            if entity:
                entities.append((" ".join(entity), label))
            entity = [word]
            label = tag[2:]
            
        elif tag.startswith("I-") and entity:
            entity.append(word)
            
        else:
            if entity:
                entities.append((" ".join(entity), label))
                entity = []
                label = None
    
    if entity:
        entities.append((" ".join(entity), label))
    
    return entities


true_entities = extract_true_entities(train_data[0])

print("Ground Truth Entities:")
print(true_entities)

In [ ]:
# spaCy predictions
predicted_entities = [(ent.text, ent.label_) for ent in doc.ents]

print("Predicted Entities:")
print(predicted_entities)

print("\nTrue Entities:")
print(true_entities)

In [ ]:
displacy.render(doc, style="ent", jupyter=True)

In [ ]:
!python -m spacy download en_core_web_md

In [ ]:
# Load medium model
nlp_md = spacy.load("en_core_web_md")

doc_md = nlp_md(train_texts[0])

# Extract entities
entities_sm = [(ent.text, ent.label_) for ent in doc.ents]
entities_md = [(ent.text, ent.label_) for ent in doc_md.ents]

print("Entities from en_core_web_sm:", entities_sm)
print("Entities from en_core_web_md:", entities_md)

In [ ]:
all_entities = []

for sentence in train_texts[:100]:  # first 100 sentences for speed
    doc = nlp_sm(sentence)
    all_entities.append([(ent.text, ent.label_) for ent in doc.ents])

# Show sample
print("Sample extracted entities from first 5 sentences:")
for i, ents in enumerate(all_entities[:5]):
    print(f"Sentence {i+1}:", ents)

In [ ]:
def flatten_entities(sentences, nlp_model):
    """
    Convert list of sentences to a list of sets of predicted entities.
    Each sentence -> set of (entity_text, label)
    """
    all_entities = []
    for sentence in sentences:
        doc = nlp_model(sentence)
        ents = set((ent.text, ent.label_) for ent in doc.ents)
        all_entities.append(ents)
    return all_entities

In [ ]:
def flatten_true_entities(sentences):
    """
    Convert list of sentences in CoNLL format to sets of true entities.
    Each sentence -> set of (entity_text, label)
    """
    all_entities = []
    for sentence in sentences:
        ents = set(extract_true_entities(sentence))
        all_entities.append(ents)
    return all_entities

# Flatten ground truth for first 100 sentences for speed
true_entities_flat = flatten_true_entities(train_data[:100])

In [ ]:
# Use the small model
pred_entities_flat = flatten_entities(train_texts[:100], nlp_sm)

# Check sample
for i in range(3):
    print(f"Sentence {i+1} predicted entities:", pred_entities_flat[i])

In [ ]:
def compute_metrics(true_sets, pred_sets):
    """
    Compute overall Precision, Recall, F1 across sentences.
    """
    assert len(true_sets) == len(pred_sets)
    
    tp = 0  # true positives
    fp = 0  # false positives
    fn = 0  # false negatives
    
    for true, pred in zip(true_sets, pred_sets):
        tp += len(true & pred)
        fp += len(pred - true)
        fn += len(true - pred)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f1

precision, recall, f1 = compute_metrics(true_entities_flat, pred_entities_flat)

print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

In [ ]:
TRAIN_DATA = [
    ("EU rejects German call to boycott British lamb .", {"entities": [(0, 2, "ORG"), (11, 17, "MISC"), ...]}),
    ...
]

In [ ]:
def conll_to_spacy_format(sentences):
    """
    Convert CoNLL sentences to spaCy training format.
    """
    training_data = []
    
    for sentence in sentences:
        text = sentence_to_text(sentence)
        entities = []
        offset = 0
        
        for word, tag in sentence:
            start = text.find(word, offset)
            end = start + len(word)
            if tag != "O":
                label = tag[2:]  # remove B- or I-
                entities.append((start, end, label))
            offset = end
        
        training_data.append((text, {"entities": entities}))
    
    return training_data

# Convert first 500 sentences for training (speed)
TRAIN_DATA = conll_to_spacy_format(train_data[:500])

# Show sample
TRAIN_DATA[0]

In [ ]:
import random
from spacy.training.example import Example

# Create blank English model
nlp_train = spacy.blank("en")

# Add NER pipeline
if "ner" not in nlp_train.pipe_names:
    ner = nlp_train.add_pipe("ner")
else:
    ner = nlp_train.get_pipe("ner")

# Add labels from training data
for _, annotations in TRAIN_DATA:
    for ent in annotations.get("entities"):
        ner.add_label(ent[2])

print("NER labels added:", ner.labels)

In [ ]:
# Disable other pipes for training
other_pipes = [pipe for pipe in nlp_train.pipe_names if pipe != "ner"]
with nlp_train.disable_pipes(*other_pipes):
    
    optimizer = nlp_train.begin_training()
    n_iter = 20  # number of epochs, can increase for better performance
    
    for itn in range(n_iter):
        random.shuffle(TRAIN_DATA)
        losses = {}
        
        for text, annotations in TRAIN_DATA:
            doc = nlp_train.make_doc(text)
            example = Example.from_dict(doc, annotations)
            nlp_train.update([example], drop=0.3, losses=losses)
        
        print(f"Iteration {itn+1} — Losses: {losses}")

In [ ]:
# Test the trained model
test_sentence = train_texts[0]  # first training sentence

doc_custom = nlp_train(test_sentence)

print("Sentence:", test_sentence)
print("\nCustom NER Predictions:")

for ent in doc_custom.ents:
    print(ent.text, "→", ent.label_)

# Optional: visualize
displacy.render(doc_custom, style="ent", jupyter=True)

In [ ]:
# Limit to first 100 sentences for speed
valid_texts_subset = [sentence_to_text(s) for s in valid_data[:100]]
true_entities_valid = flatten_true_entities(valid_data[:100])

# Flatten custom NER predictions
pred_entities_custom = flatten_entities(valid_texts_subset, nlp_train)

# Show sample comparison
for i in range(3):
    print(f"Sentence {i+1} predicted:", pred_entities_custom[i])
    print(f"Sentence {i+1} true:", true_entities_valid[i])
    print("---")

In [ ]:
# Reuse the compute_metrics function
precision_c, recall_c, f1_c = compute_metrics(true_entities_valid, pred_entities_custom)

print(f"Custom NER — Precision: {precision_c:.4f}")
print(f"Custom NER — Recall:    {recall_c:.4f}")
print(f"Custom NER — F1-score:  {f1_c:.4f}")

In [ ]:
# Visualize first 5 validation sentences
from IPython.display import display, HTML

for i, sentence in enumerate(valid_texts[:5]):
    doc = nlp_train(sentence)
    print(f"Sentence {i+1}: {sentence}\nEntities:")
    for ent in doc.ents:
        print(ent.text, "→", ent.label_)
    
    # Visualize using displaCy
    html = displacy.render(doc, style="ent", jupyter=False)
    display(HTML(html))
    print("\n" + "-"*80 + "\n")